# 탐지 데이터 시작하기

`DETECTOR_DATA_GUIDE.md`의 데이터 배치 안내에 따라
공유받은 자료를 로컬 저장소의 `datasets/`에 넣으세요.

- `datasets/images/select_aihub/`: AI Hub 이미지 파일
- `datasets/images/select_codeit/`: 대회 이미지 파일
- `datasets/annotations/`: `train.json`, `validation.json`
- `datasets/`: `split_manifest.csv`, `class_mapping.csv`

`datasets/`는 로컬에만 보관하며 GitHub에는 업로드하지 않습니다.

팀 저장소의 학습 환경을 Notebook 커널로 선택하세요.
이 Notebook은 검사와 DataLoader 준비까지만 수행합니다.
실제 학습은 마지막 안내의 별도 셀에서 시작합니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in candidates
     if (path / "detector_data").is_dir() and (path / "detector").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("공유 저장소 안에서 이 Notebook을 실행하세요.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("저장소:", project_root)
print("Python:", sys.executable)


## 1. 전체 라벨·분할·파일 경로 검사


In [ ]:
subprocess.run(
    [sys.executable, "-m", "detector_data", "check"], check=True
)

## 2. 실제 이미지 소량 입력 검사

Train 2장과 Validation 2장을 읽습니다. 전체 데이터에서 다시 분할하지 않습니다.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "detector_data", "smoke"], check=True
)

## 3. 전체 학습 입력 준비

이 셀은 전체 DataLoader를 생성하고 검증합니다. 학습은 시작하지 않습니다.
기존 학습 결과가 있는 경우 `run_name`을 바꿔 주세요.


In [ ]:
from detector_data.start import prepare_training_inputs

config, device, train_loader, val_loader, label_map, data_summary = (
    prepare_training_inputs(
        run_name="pill_baseline_01",
        batch_size=8,
        seed=42,
        max_epochs=50,
    )
)

## 다음 단계

모델 순전파·역전파까지 소량으로 확인하려면 저장소 터미널에서 실행합니다.

새 터미널에서는 먼저 저장소의 학습 환경을 활성화하세요. Notebook 커널을 선택해도 터미널 환경은 자동으로 활성화되지 않습니다. Mac/Linux에서는 `source .venv/bin/activate`를 실행한 뒤 아래 검사 명령을 실행합니다.

```bash
source .venv/bin/activate
python -m detector_data smoke --model-step --report outputs/detector_data/model_smoke_report.json
```

**아래 예시부터 실제 전체 학습을 시작합니다.** 스모크 테스트만 하려면 실행하지 않습니다. 실제 학습을 시작하기로 결정했을 때만 같은 Notebook의 새 코드 셀에 복사합니다.

```python
from detector import run_training
result = run_training(
    config, train_loader, val_loader, label_map, data_summary,
    device=device,
)
```

이 학습 호출은 사전 학습 backbone 가중치를 처음 다운로드할 수 있습니다.
